In [1]:
import os
import numpy as np
import pandas as pd
datadir = os.path.join('..', 'stock-market-dataset')
outdir = os.path.join('..', 'DELAY', 'recalls-spillover-data')
refNetwork = pd.read_csv(os.path.join(outdir, 'refNetwork.csv'))

In [2]:
start_date, end_date, fn = '2013-01-01', '2022-12-31', 'stocks_adj_close.csv'
X = pd.read_csv(os.path.join(datadir, fn), index_col = 0, parse_dates = True)
cols = refNetwork.Gene2.sort_values().unique()
X = np.log1p(X.loc[start_date : end_date, cols].bfill())
X.T.to_csv(os.path.join(outdir, 'NormalizedData.csv')); X

,ABEO,ABUS,ACHV,ACIU,ADPT,AGEN,AHCO,AIM,AIMD,AKBA,...,VYNE,WST,XAIR,XBIT,XGN,XOMA,XTNT,ZLAB,ZYME,ZYXI
2013-01-02,5.824524,1.801710,10.290755,2.813011,3.720862,4.433419,2.370244,9.488048,4.066888,3.321252,...,7.634395,3.289158,4.094345,3.046109,2.896464,3.966511,5.048573,3.364879,2.639057,0.294477
2013-01-03,5.786897,1.783391,10.287014,2.813011,3.720862,4.412221,2.370244,9.447229,4.066888,3.321252,...,7.634395,3.292638,4.094345,3.046109,2.896464,4.014580,5.063860,3.364879,2.639057,0.294477
2013-01-04,5.747799,1.803359,10.277978,2.813011,3.720862,4.419337,2.370244,9.565003,4.066888,3.321252,...,7.634395,3.295933,4.094345,3.046109,2.896464,4.025352,5.101085,3.364879,2.639057,0.276089
2013-01-07,5.747799,1.788421,10.269623,2.813011,3.720862,4.405054,2.370244,9.601368,4.066888,3.321252,...,7.634395,3.290029,4.094345,3.046109,2.896464,4.074142,5.071417,3.364879,2.639057,0.276089
2013-01-08,5.747799,1.796747,10.250370,2.813011,3.720862,4.397836,2.370244,9.565003,4.066888,3.321252,...,7.634395,3.287588,4.094345,3.046109,2.896464,4.097672,5.078917,3.364879,2.639057,0.276089
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2022-12-23,1.255616,1.220830,1.252763,1.009781,2.140066,3.893581,3.002211,3.637586,2.815409,0.355574,...,1.144860,5.457244,4.834693,1.406097,1.238374,2.917771,0.506818,3.458837,2.201659,2.695303
2022-12-27,1.226712,1.202972,1.229641,1.007958,2.098018,3.840201,2.997730,3.583519,2.815409,0.345007,...,1.237794,5.463997,4.780803,1.393766,1.217876,2.931727,0.506818,3.417399,2.138889,2.691921
2022-12-28,1.223775,1.169381,1.232560,1.006131,2.068128,3.774846,3.001714,3.526361,2.847812,0.352767,...,1.258461,5.440633,4.789157,1.420696,1.220830,2.930660,0.488580,3.400197,2.138889,2.700018
2022-12-29,1.340250,1.199965,1.266948,1.040277,2.148268,3.835975,3.016515,3.465736,2.819890,0.409457,...,1.243001,5.473416,4.865995,1.474763,1.229641,2.966303,0.506818,3.454106,2.175887,2.699346


In [3]:
t = np.arange(X.shape[0], dtype = float)
t = pd.Series(t / t.max(), index = X.index, name = 'PseudoTime')
t.to_csv(os.path.join(outdir, 'PseudoTime.csv')); t

2013-01-02    0.000000
2013-01-03    0.000397
2013-01-04    0.000795
2013-01-07    0.001192
2013-01-08    0.001589
                ...   
2022-12-23    0.998411
2022-12-27    0.998808
2022-12-28    0.999205
2022-12-29    0.999603
2022-12-30    1.000000
Name: PseudoTime, Length: 2518, dtype: float64

In [4]:
fn = os.path.join(outdir, 'TranscriptionFactors.csv')
symbols = pd.Series(X.columns)
symbols.to_csv(fn, index = False, header = False)
symbols

0      ABEO
1      ABUS
2      ACHV
3      ACIU
4      ADPT
       ... 
286    XOMA
287    XTNT
288    ZLAB
289    ZYME
290    ZYXI
Length: 291, dtype: object

In [5]:
N = X.shape[1]
C = refNetwork.groupby('Gene1').size() / N
C.sort_values(ascending = False, inplace = True)
print(C, '\n\n', C.mean())

Gene1
ICUI    0.274914
BAX     0.178694
PHG     0.147766
BSX     0.109966
CAH     0.106529
VTRS    0.096220
NEPH    0.082474
MRK     0.068729
PODD    0.065292
FMS     0.058419
BDX     0.037801
EW      0.037801
WST     0.037801
OSUR    0.020619
dtype: float64 

 0.09450171821305843


In [6]:
val_symbols = ['PHG', 'EW']
fold = pd.Series(1, index = C.index, name = 'Split')
fold.loc[val_symbols] = 2
fold.to_csv(os.path.join(outdir, 'splitLabels.csv'))
fold

Gene1
ICUI    1
BAX     1
PHG     2
BSX     1
CAH     1
VTRS    1
NEPH    1
MRK     1
PODD    1
FMS     1
BDX     1
EW      2
WST     1
OSUR    1
Name: Split, dtype: int64

In [7]:
print([(i, C.loc[fold == i].mean()) for i in (1, 2)])

[(1, 0.0947880870561283), (2, 0.09278350515463918)]
